# Unit Tests for `assessment_processor.py`

Run the cells in this notebook to test key parser behavior with small temporary DOCX and HTML fixtures. 

In [1]:
from pathlib import Path
import sys
import tempfile

from bs4 import BeautifulSoup
from docx import Document


def find_preprocessing_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "assessment_processor.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate preprocessing/assessment_processor.py")


preprocessing_dir = find_preprocessing_dir(Path.cwd().resolve())
if str(preprocessing_dir) not in sys.path:
    sys.path.insert(0, str(preprocessing_dir))

from assessment_processor import AssessmentParser, parse_to_dict

parser = AssessmentParser()


def run_test(name, test_func):
    try:
        test_func()
    except Exception as exc:
        print(f"FAIL: {name}")
        print(f"{type(exc).__name__}: {exc}")
        raise
    else:
        print(f"PASS: {name}")

In [2]:
def cleanup(path: Path) -> None:
    try:
        path.unlink()
    except FileNotFoundError:
        pass


def make_temp_docx() -> Path:
    doc = Document()
    doc.add_heading("Main Section", level=1)
    doc.add_heading("Sub Section", level=2)

    paragraph = doc.add_paragraph()
    paragraph.add_run("Intro ")
    bold_run = paragraph.add_run("Bold")
    bold_run.bold = True
    paragraph.add_run(" ")
    italic_run = paragraph.add_run("Name")
    italic_run.italic = True

    table = doc.add_table(rows=2, cols=2)
    header_run = table.cell(0, 0).paragraphs[0].add_run("Header")
    header_run.bold = True
    table.cell(0, 1).text = "Value"
    table.cell(1, 0).text = "A"
    table.cell(1, 1).text = "B"

    temp = tempfile.NamedTemporaryFile(suffix=".docx", delete=False)
    temp.close()
    path = Path(temp.name)
    doc.save(path)
    return path


def make_temp_html(suffix=".html") -> Path:
    html = """
    <html>
      <body>
        <h1>HTML Section</h1>
        <h2>HTML Subsection</h2>
        <p>Plain <strong>bold text</strong> and <em>italic text</em></p>
        <ul><li>First item</li><li>Second item</li></ul>
        <table><tr><th>Header</th></tr><tr><td>Value</td></tr></table>
      </body>
    </html>
    """
    temp = tempfile.NamedTemporaryFile("w", suffix=suffix, delete=False, encoding="utf-8")
    temp.write(html)
    temp.close()
    return Path(temp.name)

## Test 1: Rich Text Wrapper

Checks that `_wrap_rich_text` combines superscript, italic, and bold tags in the expected order.

In [3]:
def test_wrap_rich_text():
    result = parser._wrap_rich_text("Name", bold=True, italic=True, vertical="superscript")
    assert result == "<b><i><sup>Name</sup></i></b>"


run_test("rich text wrapper", test_wrap_rich_text)

PASS: rich text wrapper


## Test 2: Paragraph Rich Text

Checks that DOCX paragraph rendering preserves subscript, superscript, bold, and italic text.

In [4]:
def test_paragraph_rich_text():
    doc = Document()
    paragraph = doc.add_paragraph()
    paragraph.add_run("CO")
    subscript_run = paragraph.add_run("2")
    subscript_run.font.subscript = True
    paragraph.add_run(" and km")
    superscript_run = paragraph.add_run("2")
    superscript_run.font.superscript = True
    bold_run = paragraph.add_run(" bold")
    bold_run.bold = True
    italic_run = paragraph.add_run(" italic")
    italic_run.italic = True

    result = parser._paragraph_text_with_scripts(paragraph)
    assert result == "CO<sub>2</sub> and km<sup>2</sup><b> bold</b><i> italic</i>"


run_test("paragraph rich text", test_paragraph_rich_text)

PASS: paragraph rich text


## Test 3: Style Bucket Merge

Checks that `_merge_style_bucket` preserves order and avoids duplicate style snippets.

In [5]:
def test_style_bucket_merge():
    target = {"bold": ["A"], "italic": []}
    source = {"bold": ["A", "B", ""], "italic": ["I", "I"]}
    parser._merge_style_bucket(target, source)

    assert target["bold"] == ["A", "B"]
    assert target["italic"] == ["I"]


run_test("style bucket merge", test_style_bucket_merge)

PASS: style bucket merge


## Test 4: Heading Detection

Checks that Heading 1 and Heading 2 paragraph styles map to the expected heading levels.

In [6]:
def test_heading_detection():
    doc = Document()
    h1 = doc.add_heading("Main", level=1)
    h2 = doc.add_heading("Sub", level=2)
    normal = doc.add_paragraph("Body")

    assert parser._heading_level(h1) == 1
    assert parser._heading_level(h2) == 2
    assert parser._heading_level(normal) is None


run_test("heading detection", test_heading_detection)

PASS: heading detection


## Test 5: Table Extraction

Checks that DOCX tables are extracted as plain rows and rich-text rows.

In [7]:
def test_table_extraction():
    doc = Document()
    table = doc.add_table(rows=2, cols=2)
    header_run = table.cell(0, 0).paragraphs[0].add_run("Header")
    header_run.bold = True
    table.cell(0, 1).text = "Value"
    table.cell(1, 0).text = "A"
    table.cell(1, 1).text = "B"

    assert parser._table_to_rows(table) == [["Header", "Value"], ["A", "B"]]
    assert parser._table_to_rows_rich(table) == [["<b>Header</b>", "Value"], ["A", "B"]]


run_test("table extraction", test_table_extraction)

PASS: table extraction


## Test 6: Full DOCX Parsing

Checks that a temporary DOCX file is parsed into headings, paragraph blocks, table blocks, and style data.

In [8]:
def test_full_docx_parsing():
    path = make_temp_docx()
    try:
        data = parser.parse_file(str(path))
    finally:
        cleanup(path)

    assert data["title"] == path.stem
    assert data["children"][0]["title"] == "Main Section"
    subsection = data["children"][0]["children"][0]
    assert subsection["title"] == "Sub Section"
    assert subsection["blocks"][0]["text"] == "Intro Bold Name"
    assert subsection["blocks"][0]["text_rich"] == "Intro <b>Bold</b> <i>Name</i>"
    assert subsection["blocks"][1]["rows"][1] == ["A", "B"]
    assert subsection["blocks"][1]["rows_rich"][0][0] == "<b>Header</b>"
    assert "Bold" in subsection["blocks"][-1]["data"]["bold"]
    assert "Name" in subsection["blocks"][-1]["data"]["italic"]


run_test("full docx parsing", test_full_docx_parsing)

PASS: full docx parsing


## Test 7: DOCX Without Comments

Checks that a normal DOCX file without Word comments returns an empty comments list.

In [9]:
def test_docx_without_comments():
    path = make_temp_docx()
    try:
        data = parser.parse_file(str(path))
    finally:
        cleanup(path)

    assert data["comments"] == []


run_test("docx without comments", test_docx_without_comments)

PASS: docx without comments


## Test 8: HTML Parsing

Checks that a temporary HTML file is parsed into headings, paragraph, list, table, and style blocks.

In [10]:
def test_html_parsing():
    path = make_temp_html()
    try:
        data = parser.parse_file(str(path))
    finally:
        cleanup(path)

    assert data["title"] == path.stem
    assert data["comments"] == []
    assert data["children"][0]["title"] == "HTML Section"
    subsection = data["children"][0]["children"][0]
    assert subsection["title"] == "HTML Subsection"
    assert subsection["blocks"][0] == {"type": "paragraph", "text": "Plain bold text and italic text"}
    assert subsection["blocks"][1]["type"] == "list"
    assert subsection["blocks"][2]["rows"] == [["Header"], ["Value"]]
    assert subsection["blocks"][-1]["type"] == "style"


run_test("html parsing", test_html_parsing)

PASS: html parsing


## Test 9: HTML Style Extraction

Checks that tag-based, class-based, and inline-style bold/italic snippets are extracted from HTML.

In [11]:
def test_html_style_extraction():
    soup = BeautifulSoup(
        """
        <p>
          <b>Strong</b>
          <i>Emphasis</i>
          <span class="dataLabel">Label</span>
          <span style="font-weight:700">Heavy</span>
          <span style="font-style: italic">Lean</span>
        </p>
        """,
        "html.parser",
    )
    styles = parser._extract_styles_from_html_element(soup.p)

    assert "Strong" in styles["bold"]
    assert "Label" in styles["bold"]
    assert "Heavy" in styles["bold"]
    assert "Emphasis" in styles["italic"]
    assert "Lean" in styles["italic"]


run_test("html style extraction", test_html_style_extraction)

PASS: html style extraction


## Test 10: `.htm` File Support

Checks that `.htm` files are accepted through the same HTML parsing path.

In [12]:
def test_htm_extension_support():
    path = make_temp_html(suffix=".htm")
    try:
        data = parser.parse_file(str(path))
    finally:
        cleanup(path)

    assert data["children"][0]["title"] == "HTML Section"


run_test("htm extension support", test_htm_extension_support)

PASS: htm extension support


## Test 11: `parse_to_dict` Helper

Checks that the public helper function parses a supported document path.

In [13]:
def test_parse_to_dict_helper():
    path = make_temp_html()
    try:
        data = parse_to_dict(str(path))
    finally:
        cleanup(path)

    assert data["children"][0]["title"] == "HTML Section"


run_test("parse_to_dict helper", test_parse_to_dict_helper)

PASS: parse_to_dict helper


## Test 12: Raw XML Run Rendering

Checks that raw DOCX XML run rendering preserves direct bold, italic, and superscript formatting.

In [14]:
def test_xml_run_text_rendering():
    doc = Document()
    paragraph = doc.add_paragraph()
    run = paragraph.add_run("Raw")
    run.bold = True
    run.italic = True
    run.font.superscript = True

    result = parser._xml_run_text_with_scripts(run._r)
    assert result == "<b><i><sup>Raw</sup></i></b>"


run_test("xml run text rendering", test_xml_run_text_rendering)

PASS: xml run text rendering


## Test 13: Unsupported File Extension

Checks that unsupported file types raise a clear `ValueError`.

In [15]:
def test_unsupported_file_extension():
    temp = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8")
    temp.write("not supported")
    temp.close()
    path = Path(temp.name)

    try:
        try:
            parser.parse_file(str(path))
        except ValueError as exc:
            assert "Unsupported file type" in str(exc)
        else:
            raise AssertionError("Expected ValueError for unsupported file type")
    finally:
        cleanup(path)


run_test("unsupported file extension", test_unsupported_file_extension)

PASS: unsupported file extension
